# People Analytics DWH — Exploring Mart Tables

This notebook demonstrates how to query the dbt mart tables directly from DuckDB.

**Prerequisites:** Run `make pipeline` to generate data and build the dbt models.

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect("../data/people_analytics.duckdb", read_only=True)
print("Tables available:")
con.sql("SHOW ALL TABLES").show()

## 1. Headcount Overview

In [ ]:
df = con.sql("""
    SELECT report_month, department, active_headcount, new_hires, terminations
    FROM marts.fct_headcount_monthly
    ORDER BY report_month DESC, active_headcount DESC
    LIMIT 20
""").df()
df

## 2. Turnover Trends

In [ ]:
turnover = con.sql("""
    SELECT
        report_year,
        department,
        ROUND(AVG(turnover_rate), 2) AS avg_monthly_turnover,
        ROUND(AVG(turnover_rate) * 12, 2) AS annualized_turnover
    FROM marts.fct_turnover_monthly
    GROUP BY report_year, department
    ORDER BY report_year, annualized_turnover DESC
""").df()
turnover

## 3. Absenteeism by Department

In [ ]:
absences = con.sql("""
    SELECT
        department,
        ROUND(AVG(absence_rate), 2) AS avg_absence_rate,
        ROUND(AVG(sick_leave_rate), 2) AS avg_sick_rate,
        ROUND(AVG(avg_bradford_factor), 1) AS avg_bradford
    FROM marts.fct_absenteeism_monthly
    GROUP BY department
    ORDER BY avg_absence_rate DESC
""").df()
absences

## 4. Employee Dimension (Sample)

In [ ]:
employees = con.sql("""
    SELECT
        employee_id, full_name, department, tenure_band,
        salary_band, courses_completed, latest_engagement_score
    FROM marts.dim_employees
    WHERE is_active = true
    ORDER BY latest_engagement_score ASC NULLS LAST
    LIMIT 15
""").df()
employees

## 5. Recruiting Efficiency

In [ ]:
recruiting = con.sql("""
    SELECT
        department,
        SUM(total_hired) AS total_hires,
        ROUND(AVG(avg_days_to_hire), 1) AS avg_time_to_fill,
        ROUND(AVG(applicants_per_hire), 1) AS avg_applicants_per_hire
    FROM marts.fct_recruiting_pipeline
    GROUP BY department
    ORDER BY avg_time_to_fill DESC
""").df()
recruiting

## 6. Training Adoption

In [ ]:
training = con.sql("""
    SELECT
        department,
        ROUND(AVG(completion_rate) * 100, 1) AS avg_completion_pct,
        ROUND(AVG(mandatory_compliance_rate) * 100, 1) AS mandatory_compliance_pct,
        ROUND(AVG(hours_per_employee), 1) AS avg_hours_per_emp
    FROM marts.fct_training_adoption
    GROUP BY department
    ORDER BY avg_completion_pct DESC
""").df()
training

In [ ]:
con.close()
print("Done.")